In [3]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the Excel workbook
file_path = 'cap_builder.xlsx'
excel_data = pd.ExcelFile(file_path)

# Read the sheets into DataFrames
starting_cap_df = excel_data.parse('starting_cap')
prop_by_manuf_df = excel_data.parse('prop_by_manuf')

output_file_path = 'production_capacity_scenarios.xlsx'

In [8]:
#adjust vaccine capacity, be selecting vaccine, percent, and column of dataframe
def adjust_capacity(df, vaccines, percentages, column):
    for vaccine, percentage in zip(vaccines, percentages):
        df.loc[df['Vaccine'] == vaccine, column] *= (1 - percentage / 100)
    return df


def scale_and_merge(proportion_df, scenario_data):
    # Merge the two DataFrames on the 'Vaccine' column
    merged_df = pd.merge(proportion_df, scenario_data, how='left', on='Vaccine')
    
    # Selecting columns to scale (excluding 'Manufacturer', 'Proportion', and 'Vaccine')
    cols_to_scale = merged_df.columns.difference(['Manufacturer', 'Proportion', 'Vaccine'])

    # Initialize the MinMaxScaler
    scaler = MinMaxScaler(feature_range=(1, 10))

    # Multiply each column (1-10) by the 'Proportion' column
    merged_df[cols_to_scale] = merged_df[cols_to_scale].multiply(merged_df['Proportion'], axis=0)

    # Round the scaled values to maintain integer-like results
    merged_df[cols_to_scale] = merged_df[cols_to_scale].round()

    

     # Aggregate by 'Manufacturer' while keeping all columns
    agg_df = merged_df.groupby('Manufacturer').agg({
        'Proportion': 'first',  # Assuming 'Proportion' is consistent within each 'Manufacturer'
        'Vaccine': 'first',     # Assuming 'Vaccine' is consistent within each 'Manufacturer'
        **{col: 'sum' for col in cols_to_scale}
    }).reset_index()

    # Drop 'Vaccine' and 'Proportion' columns
    agg_df.drop(['Vaccine', 'Proportion'], axis=1, inplace=True)

    return agg_df

In [12]:
#create base capacity, by antigen, for a ten year period
base_capacity = pd.concat([starting_cap_df["Predicted Capacity"]] * 10, axis=1)
base_capacity.columns = range(1, 11)
base_capacity = pd.concat([starting_cap_df[['Vaccine']], base_capacity], axis=1)

#create scaled capacity
base_capacity_scaled = base_capacity.copy()
base_capacity_scaled.iloc[:, 1:] *= 1.5

# base_capacity_scaled.head()

In [14]:
#create base capacity merged and save
base_df = scale_and_merge(prop_by_manuf_df, base_capacity)
# base_df.head()

base_df.to_excel(output_file_path, sheet_name='master_capacity', index=False)


In [16]:
base_df_scaled = scale_and_merge(prop_by_manuf_df, base_capacity_scaled)

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    base_df_scaled.to_excel(writer, sheet_name='base_capacity_scaled', index=False)

In [19]:
#IPV shortage scenario
vaccines = ['IPV']
# percentages = [8]

global_shortage_IPV_scaled = adjust_capacity(base_capacity_scaled, vaccines, [8], 3)
global_shortage_IPV_scaled = adjust_capacity(global_shortage_IPV_scaled, vaccines, [10], 4)
global_shortage_IPV_scaled = adjust_capacity(global_shortage_IPV_scaled, vaccines, [3], 5)
global_shortage_IPV_scaled = scale_and_merge(prop_by_manuf_df, global_shortage_IPV_scaled)
# global_shortage_IPV

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    global_shortage_IPV_scaled.to_excel(writer, sheet_name='IPV_Shortage_scaled', index=False)


In [4]:
#IPV shortage scenario
vaccines = ['IPV']
# percentages = [8]

global_shortage_IPV = adjust_capacity(base_capacity, vaccines, [8], 3)
global_shortage_IPV = adjust_capacity(global_shortage_IPV, vaccines, [10], 4)
global_shortage_IPV = adjust_capacity(global_shortage_IPV, vaccines, [3], 5)
global_shortage_IPV = scale_and_merge(prop_by_manuf_df, global_shortage_IPV)
# global_shortage_IPV

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    global_shortage_IPV.to_excel(writer, sheet_name='IPV_Shortage', index=False)

In [28]:
base_capacity.Vaccine.unique()

array(['DT', 'DTwP', 'DTwP-Hib', 'HepB', 'Hexa', 'Hib', 'HPV', 'IPV',
       'Measles', 'MMR', 'MR', 'OPV', 'PCV', 'Penta', 'Rota', 'Td', 'TT'],
      dtype=object)

In [24]:
#pandemic scenario
vaccines = base_capacity.Vaccine.unique()
# percentages = [44]
# columns = [4]

pandemic_df = adjust_capacity(base_capacity, vaccines, [44], [4])
pandemic_df = adjust_capacity(pandemic_df, vaccines, [35], [5])
pandemic_df = adjust_capacity(pandemic_df, vaccines, [6], [6])
pandemic_df = scale_and_merge(prop_by_manuf_df, pandemic_df)
# global_shortage_IPV

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    pandemic_df.to_excel(writer, sheet_name='pandemic', index=False)

In [26]:
#pandemic scenario scaled
vaccines = base_capacity.Vaccine.unique()
# percentages = [44]
# columns = [4]

pandemic_df_scaled = adjust_capacity(base_capacity_scaled, vaccines, [44], [4])
pandemic_df_scaled = adjust_capacity(pandemic_df_scaled, vaccines, [35], [5])
pandemic_df_scaled = adjust_capacity(pandemic_df_scaled, vaccines, [6], [6])
pandemic_df_scaled = scale_and_merge(prop_by_manuf_df, pandemic_df_scaled)
# global_shortage_IPV

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    pandemic_df_scaled.to_excel(writer, sheet_name='pandemic_scaled', index=False)

In [30]:
#funding delay
vaccines1 = ['DT', 'IPV', 'Measles', 'Td', 'TT']
vaccines2 = ['Hexa', 'OPV']
vaccines3 = ['Measles', 'MMR', 'MR']
vaccines4 = ['DTwP', 'DTwP-Hib', 'Hexa', 'Penta', 'Rota']
percentage = [8]

funding_delay = adjust_capacity(base_capacity, vaccines1, percentage, [1])
funding_delay = adjust_capacity(funding_delay, vaccines2, percentage, [3])
funding_delay = adjust_capacity(funding_delay, vaccines3, percentage, [5])
funding_delay = adjust_capacity(funding_delay, vaccines4, percentage, [6])
funding_delay = scale_and_merge(prop_by_manuf_df, funding_delay)

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    funding_delay.to_excel(writer, sheet_name='funding_delay', index=False)

In [32]:
#funding delay scaled
vaccines1 = ['DT', 'IPV', 'Measles', 'Td', 'TT']
vaccines2 = ['Hexa', 'OPV']
vaccines3 = ['Measles', 'MMR', 'MR']
vaccines4 = ['DTwP', 'DTwP-Hib', 'Hexa', 'Penta', 'Rota']
percentage = [8]

funding_delay_scaled = adjust_capacity(base_capacity_scaled, vaccines1, percentage, [1])
funding_delay_scaled = adjust_capacity(funding_delay_scaled, vaccines2, percentage, [3])
funding_delay_scaled = adjust_capacity(funding_delay_scaled, vaccines3, percentage, [5])
funding_delay_scaled = adjust_capacity(funding_delay_scaled, vaccines4, percentage, [6])
funding_delay_scaled = scale_and_merge(prop_by_manuf_df, funding_delay_scaled)

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    funding_delay_scaled.to_excel(writer, sheet_name='funding_delay_scaled', index=False)

In [ ]:
#innacurate forecast
vaccines1 = ['HepB']
vaccines2 = ['DTwP', 'DTwP-Hib', 'Hexa', 'Penta']
vaccines3 = ['Measles', 'MMR', 'MR']
vaccines4 = ['DTwP', 'DTwP-Hib', 'Hexa', 'Penta', 'Rota']
percentage = [8]

funding_delay_scaled = adjust_capacity(base_capacity_scaled, vaccines1, percentage, [1])
funding_delay_scaled = adjust_capacity(funding_delay_scaled, vaccines2, percentage, [3])
funding_delay_scaled = adjust_capacity(funding_delay_scaled, vaccines3, percentage, [5])
funding_delay_scaled = adjust_capacity(funding_delay_scaled, vaccines4, percentage, [6])
funding_delay_scaled = scale_and_merge(prop_by_manuf_df, funding_delay_scaled)

with pd.ExcelWriter(output_file_path, engine='openpyxl', mode='a') as writer:
    funding_delay_scaled.to_excel(writer, sheet_name='funding_delay_scaled', index=False)

In [ ]:
['DT', 'DTwP', 'DTwP-Hib', 'HepB', 'Hexa', 'Hib', 'HPV', 'IPV',
       'Measles', 'MMR', 'MR', 'OPV', 'PCV', 'Penta', 'Rota', 'Td', 'TT']